In [0]:
%sql
CREATE OR REPLACE TABLE electronics_retailer_clg.gold.fact_sales AS
SELECT 
  s.order_number,
  s.order_date,
  s.delivery_date,
  DATEDIFF(s.delivery_date, s.order_date) as delivery_time_days,
  s.customerkey,
  c.gender,
  c.continent,
  s.storekey,
  st.country as store_country,
  st.channel,
  s.productkey,
  p.category as product_category,
  p.unit_price_usd,
  s.quantity,
  s.currency_code,
  er.exchange as exchange_rate,
  -- Revenue calculation in USD
  (s.quantity * p.unit_price_usd * COALESCE(er.exchange, 1.0)) as revenue_usd
FROM electronics_retailer_clg.silver.sales s
LEFT JOIN electronics_retailer_clg.gold.dim_customers c ON s.customerkey = c.customerkey
LEFT JOIN electronics_retailer_clg.gold.dim_products p ON s.productkey = p.productkey
LEFT JOIN electronics_retailer_clg.gold.dim_stores st ON s.storekey = st.storekey
LEFT JOIN electronics_retailer_clg.silver.exchange_rates er ON s.currency_code = er.currency
WHERE s.order_date IS NOT NULL;

In [0]:
df = spark.table("electronics_retailer_clg.gold.fact_sales").filter("year(order_date) = 2019").orderBy("order_date").select("order_date")
display(df)